In [2]:
# Imports and setup
import sys
from pathlib import Path
# Ensure the repository `src` folder is on sys.path so `from utils...` works when running cells
sys.path.append(str(Path('../../src').resolve()))

import pandas as pd, numpy as np, time, traceback, ast
from pathlib import Path
from joblib import dump
from sklearn.base import clone
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import matplotlib.dates as mdates



from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

from utils.models import MODELS
from utils.eval_metrics import evaluate_model
from utils.variants import VARIANT_DEFS, apply_variant, prepare_variant_data_full


# Paths
BASE_DATA_PATH = Path("../../data/out/dataset_final.csv")
VALIDATION_PATH = Path("../../data/out/dataset_validation_final.csv")
BEST_MODELS = Path("../../data/out/best_models_ml/in/best_models_ml.xlsx")

OUT_DIR = Path("../../data/out/best_models_ml/validation_ml")
MODELS_DIR = OUT_DIR / "models"
PLOTS_DIR = OUT_DIR / "plots"
PRED_DIR  = OUT_DIR / "predictions" 
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)
PRED_DIR.mkdir(exist_ok=True)



TARGET_COLUMN = "PRECIO"
categorical_numeric = ["YEAR","MONTH","DAY","HORA","NIVEL_ENSO","DIA_SEMANA","FESTIVO"]

In [3]:
# %% Load datasets
df_train = pd.read_csv(BASE_DATA_PATH)
df_val = pd.read_csv(VALIDATION_PATH)
best_df = pd.read_excel(BEST_MODELS)

In [4]:
rows = []

for _, row in best_df.iterrows():
    variant = row["variant"]       # ej. "v1_original"
    model_name = row["model"]      # ej. "RandomForestRegressor"

    # Parsear best_params desde Excel
    try:
        best_params = ast.literal_eval(str(row.get("best_params", "{}")))
        if not isinstance(best_params, dict):
            best_params = {}
    except Exception:
        best_params = {}

    try:
        # --- Aplicar la variante al dataset ---
        vdef = next(v for v in VARIANT_DEFS if v["name"] == variant)
        df_train_var, _ = apply_variant(df_train.copy(), vdef, date_col="FECHA_HORA")
        df_val_var, _   = apply_variant(df_val.copy(), vdef, date_col="FECHA_HORA")

        # --- Preparar datos ---
        preprocessor, X_full, y_full = prepare_variant_data_full(df_train_var)
        _, X_val, y_val = prepare_variant_data_full(df_val_var)

        # --- Clonar modelo base y setear hiperparámetros ---
        base_estimator = MODELS[model_name]
        estimator = clone(base_estimator).set_params(**best_params)

        # --- Pipeline: preprocesador + modelo ---
        pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])

        # --- Entrenar con 100% del dataset ---
        start = time.time()
        pipe.fit(X_full, y_full)
        elapsed = time.time() - start

        # --- Predecir en validación externa ---
        y_pred = pipe.predict(X_val)
        metrics = evaluate_model(pipe, X_val, y_val)
        metrics.update({
            "variant": variant,
            "model": model_name,
            "train_time_s": elapsed,
            "n_train": len(y_full),
            "n_val": len(y_val),
        })
        rows.append(metrics)

        # --- Guardar modelo entrenado ---
        dump(pipe, MODELS_DIR / f"{variant}_{model_name}.joblib")

        # --- Guardar predicciones ---
        df_preds = pd.DataFrame({
            "y_real": y_val,
            "y_pred": y_pred
        })
        df_preds.to_csv(PRED_DIR / f"predictions_{variant}_{model_name}.csv", index=False)

        # --- Gráficas ---
        n = len(y_val)
        n_observations = 24 * 7  # 7 días
        fechas = pd.date_range(start="2025-07-01 00:00", periods=n, freq="H")

        # 1. Predicciones vs reales
        plt.figure(figsize=(14,5))
        plt.plot(fechas[:n_observations], y_val.values[:n_observations], label="Real")
        plt.plot(fechas[:n_observations], y_pred[:n_observations], label="Predicho")
        plt.xlabel("Fecha y hora")
        plt.ylabel("Precio spot (COP/kWh)")
        plt.legend()
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%d-%b %Hh"))
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"pred_{variant}_{model_name}.png")
        plt.close()

        # 2. Scatter con línea y=x
        plt.figure(figsize=(5,5))
        plt.scatter(y_val, y_pred, s=6, alpha=0.5)
        lims = [min(plt.xlim()[0], plt.ylim()[0]), max(plt.xlim()[1], plt.ylim()[1])]
        plt.plot(lims, lims, 'r--', linewidth=2)
        plt.xlabel("Valores Reales Precio Spot (COP/kWh)")
        plt.ylabel("Valores Predichos Precio Spot (COP/kWh)")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"scatter_{variant}_{model_name}.png")
        plt.close()

        # 3. Residuos
        resid = y_val - y_pred
        plt.figure(figsize=(14,4))
        plt.plot(fechas[:n_observations], resid[:n_observations], color="tab:red")
        plt.xlabel("Fecha y hora")
        plt.ylabel("Residuo Precio (COP/kWh) (Real-Predicho)")
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%d-%b %Hh"))
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"resid_{variant}_{model_name}.png")
        plt.close()

        print(f"OK {variant}-{model_name}: MAE={metrics.get('MAE'):.2f}")

    except Exception as e:
        print(f"ERROR {variant}-{model_name}: {e}")
        traceback.print_exc()

# --- Guardar métricas consolidadas ---
if rows:
    df_out = pd.DataFrame(rows)
    df_out.to_excel(OUT_DIR / "validation_ml_metrics.xlsx", index=False)

C:\Users\Camilo\AppData\Local\Temp\ipykernel_20412\2225914410.py:62: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  fechas = pd.date_range(start="2025-07-01 00:00", periods=n, freq="H")


OK v1_original-ElasticNet: MAE=150.42


C:\Users\Camilo\AppData\Local\Temp\ipykernel_20412\2225914410.py:62: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  fechas = pd.date_range(start="2025-07-01 00:00", periods=n, freq="H")


OK v1_original_lags-Lasso: MAE=28.32


C:\Users\Camilo\AppData\Local\Temp\ipykernel_20412\2225914410.py:62: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  fechas = pd.date_range(start="2025-07-01 00:00", periods=n, freq="H")


...

OK v7_only_gen_enso_lags-Linear Regression: MAE=117.29


C:\Users\Camilo\AppData\Local\Temp\ipykernel_20412\2225914410.py:62: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  fechas = pd.date_range(start="2025-07-01 00:00", periods=n, freq="H")


OK v8_only_gen_no_solar_enso-Linear Regression: MAE=117.29


C:\Users\Camilo\AppData\Local\Temp\ipykernel_20412\2225914410.py:62: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  fechas = pd.date_range(start="2025-07-01 00:00", periods=n, freq="H")


OK v8_only_gen_no_solar_enso_lags-XGBoost: MAE=94.85
